## Limpieza de 'DF_YOUEVENT_SUCIO.csv'

`scrape_youevent.ipynb` ya hace todo el trabajo de red en dos fases: la Fase 1 produce `DF_YOUEVENT_SUCIO.csv` (una fila por enlace de clasificación, 1.858 eventos, 9.013 enlaces, con `sexo` clasificado por la etiqueta del enlace) y la Fase 2 produce `DF_YOUEVENT_FINISHERS_SUCIO.csv` (una fila por PDF, con el recuento real de corredores por sexo, sacado del código de categoría dentro de cada PDF -- no de la etiqueta, por eso cubre mucho más que el 23,5 % de enlaces con etiqueta clara). Este notebook no descarga nada: solo carga esos dos CSV y los agrega a nivel de evento.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/youevent/DF_YOUEVENT_SUCIO.csv")
df = pd.read_csv(CSV_PATH)

print("Filas x columnas:", df.shape)
print("Eventos distintos:", df["nombre_evento"].nunique())
df.head()

Filas x columnas: (9021, 5)
Eventos distintos: 1860


,fecha,nombre_evento,etiqueta_clasificacion,sexo,url_pdf
0,08/03/2008,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),Clubes,NaN,https://youevent.es/sport/multimedia/clasifica...
1,08/03/2008,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),Masculina,masculino,https://youevent.es/sport/multimedia/clasifica...
2,08/03/2008,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),Femenina,femenino,https://youevent.es/sport/multimedia/clasifica...
3,23/03/2008,DUATLÓN CROS PORTILLO (2ª Circ Vallad),Clubes,NaN,https://youevent.es/sport/multimedia/clasifica...
4,23/03/2008,DUATLÓN CROS PORTILLO (2ª Circ Vallad),Masculina,masculino,https://youevent.es/sport/multimedia/clasifica...


### Tabla a nivel de evento

`df` es una fila por enlace (9.013 filas, 1.858 eventos); el esquema común del proyecto es una fila por evento. Agregamos primero (`nombre_evento`+`fecha` como clave de evento) y añadimos `tiene_sexo_identificado` y `n_enlaces_sexo` como diagnóstico -- ojo, esto mide solo la cobertura de la ETIQUETA del enlace (el 23,5 %), no la cobertura real de `DF_YOUEVENT_FINISHERS_SUCIO.csv` (que sale del código de categoría dentro de cada PDF y cubre mucho más -- ver el caso de Cantimpalos en `scrape_youevent.ipynb`). Lo dejamos igualmente porque es gratis (no hace falta abrir nada) y sirve de referencia rápida. Como en RaceResult, nunca rellenamos `finisher_d`/`finisher_h` a 0 para un evento que simplemente no se ha procesado todavía.

In [2]:
curses_limpio = df[["nombre_evento", "fecha"]].drop_duplicates().rename(
    columns={"nombre_evento": "nombre_carrera"}
).reset_index(drop=True)

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"], format="%d/%m/%Y", errors="coerce")

_cobertura = df.groupby("nombre_evento")["sexo"].apply(lambda s: s.notna().sum()).rename("n_enlaces_sexo")
curses_limpio = curses_limpio.merge(_cobertura, left_on="nombre_carrera", right_index=True, how="left")
curses_limpio["tiene_sexo_identificado"] = curses_limpio["n_enlaces_sexo"] > 0

print(curses_limpio.shape)
print("Fechas sin parsear:", curses_limpio["fecha"].isna().sum())
curses_limpio.head()

(1860, 4)
Fechas sin parsear: 0


,nombre_carrera,fecha,n_enlaces_sexo,tiene_sexo_identificado
0,DUATLON CROS MEDINA DE RIOSECO (1ª Circ Vallad),2008-03-08,2,True
1,DUATLÓN CROS PORTILLO (2ª Circ Vallad),2008-03-23,2,True
2,DUATLÓN PEÑAFIEL,2008-03-29,2,True
3,DUATLÓN CROS SIMANCAS (3ª Circ Vallad),2008-04-06,2,True
4,DUATLÓN CIUDAD DE SALAMANCA,2008-04-13,2,True


### `finisher_d`/`finisher_h`: agregando `DF_YOUEVENT_FINISHERS_SUCIO.csv`

Ese CSV lo produce la Fase 2 de `scrape_youevent.ipynb` (una fila por PDF, con `n_masculino`/`n_femenino` ya contados a partir del código de categoría). Aquí solo hace falta sumarlo por evento y hacer `merge` -- nada de red, es pura agregación. Si todavía no has corrido la Fase 2 en tu máquina (o la has corrido solo parcialmente), el fichero puede no existir o no cubrir todos los eventos: en ese caso los eventos que falten se quedan en `NaN`, nunca en 0, para no confundir "sin procesar" con "cero finishers".

In [3]:
import re

RUTA_FINISHERS = Path("../../data/raw/youevent/DF_YOUEVENT_FINISHERS_SUCIO.csv")

_RE_LOCALES = re.compile(r"local", re.IGNORECASE)
_RE_CLUBES = re.compile(r"club", re.IGNORECASE)
_RE_ORDEN_LLEGADA = re.compile(r"orden.*llegada", re.IGNORECASE)
_RE_ABSOLUTA = re.compile(r"absolut", re.IGNORECASE)
_RE_PREFIJO_DISTANCIA = re.compile(r"^\s*(\d+(?:[.,]\d+)?\s*k(?:m|ms)?)\b", re.IGNORECASE)


def _grupo_distancia(etiqueta) -> str:
    """Agrupa los PDF de un mismo evento por la distancia que llevan como
    prefijo en su etiqueta ('10KM - ...', '5km- ...'): eventos con dos
    carreras (10K y 5K) tienen dos grupos independientes, cada uno con su
    propio 'Orden de Llegada'/'Absoluta'/'Categorías' -- sumar sus totales
    finales SÍ es correcto (son corredores distintos). Las etiquetas sin
    prefijo de distancia (carreras infantiles sueltas, o eventos de una sola
    distancia que no la repiten en cada etiqueta) van todas al mismo grupo
    comodín."""
    if not isinstance(etiqueta, str):
        return "_sin_distancia"
    m = _RE_PREFIJO_DISTANCIA.match(etiqueta)
    return m.group(1).lower().replace(" ", "") if m else "_sin_distancia"


def _tipo_clasificacion(etiqueta) -> str:
    """Clasifica cada PDF por lo que representa DENTRO de su grupo de
    distancia. 'locales' y 'clubes' son SUBCONJUNTOS de los mismos corredores
    que ya cuentan 'orden_llegada'/'absoluta' (solo vecinos, o solo socios de
    club) -- nunca deben sumarse junto a ellos o contaríamos a la misma gente
    más de una vez."""
    if not isinstance(etiqueta, str):
        return "otro"
    if _RE_LOCALES.search(etiqueta):
        return "locales"
    if _RE_CLUBES.search(etiqueta):
        return "clubes"
    if _RE_ORDEN_LLEGADA.search(etiqueta):
        return "orden_llegada"
    if _RE_ABSOLUTA.search(etiqueta):
        return "absoluta"
    return "otro"


def _elegir_canonico(grupo: pd.DataFrame) -> dict:
    """De todos los PDF de un mismo (evento, grupo de distancia), decide
    cuáles sumar para tener el total de finishers UNA sola vez: prioridad
    'Orden de Llegada' > 'Absoluta' > el resto ('Categorías', carreras
    infantiles sueltas...) -- estas últimas se suman entre sí porque
    reparten a los corredores sin solaparse (cada uno en una sola categoría/
    carrera), pero solo se usan si no hay ninguna de las dos anteriores.
    'locales' y 'clubes' quedan SIEMPRE excluidos; si un grupo de distancia
    solo tiene ese tipo de PDF (ningún PDF con el campo completo), se
    descarta entero antes que arriesgarse a llamar "total" a una parte."""
    utilizables = grupo[~grupo["tipo"].isin(["locales", "clubes"])]
    if utilizables.empty:
        return {"n_masculino": 0, "n_femenino": 0, "n_pdf_usados": 0}

    for tipo in ("orden_llegada", "absoluta"):
        seleccion = utilizables[utilizables["tipo"] == tipo]
        if not seleccion.empty:
            return {
                "n_masculino": seleccion["n_masculino"].sum(),
                "n_femenino": seleccion["n_femenino"].sum(),
                "n_pdf_usados": len(seleccion),
            }

    return {
        "n_masculino": utilizables["n_masculino"].sum(),
        "n_femenino": utilizables["n_femenino"].sum(),
        "n_pdf_usados": len(utilizables),
    }


if RUTA_FINISHERS.exists():
    df_finishers_pdf = pd.read_csv(RUTA_FINISHERS)

    # La Fase 2 solo guarda url_pdf + conteos; la etiqueta de texto de cada
    # PDF (la necesitamos para saber si es "Orden de Llegada", "Locales"...)
    # vive en el dump de la Fase 1, 'df'.
    df_finishers_pdf = df_finishers_pdf.merge(
        df[["url_pdf", "etiqueta_clasificacion"]].drop_duplicates("url_pdf"),
        on="url_pdf", how="left",
    )
    df_finishers_pdf["grupo_distancia"] = df_finishers_pdf["etiqueta_clasificacion"].apply(_grupo_distancia)
    df_finishers_pdf["tipo"] = df_finishers_pdf["etiqueta_clasificacion"].apply(_tipo_clasificacion)

    _filas_canonicas = [
        {"nombre_evento": evento, "grupo_distancia": grupo_dist, **_elegir_canonico(sub)}
        for (evento, grupo_dist), sub in df_finishers_pdf.groupby(["nombre_evento", "grupo_distancia"])
    ]
    _por_grupo = pd.DataFrame(_filas_canonicas)

    _cobertura = _por_grupo.groupby("nombre_evento")["n_pdf_usados"].agg(
        total_pdf_usados="sum", n_grupos="count", n_grupos_sin_pdf=lambda s: (s == 0).sum(),
    )
    _sin_datos = _cobertura.index[_cobertura["total_pdf_usados"] == 0]
    _cobertura_parcial = _cobertura.index[
        (_cobertura["n_grupos_sin_pdf"] > 0) & (_cobertura["total_pdf_usados"] > 0)
    ]
    if len(_sin_datos):
        print(f"AVISO: {len(_sin_datos)} eventos solo tenían PDF de locales/clubes "
              "(ningún PDF con el campo completo) -- se quedan sin finisher_d/finisher_h.")
    if len(_cobertura_parcial):
        print(f"AVISO: {len(_cobertura_parcial)} eventos con varias distancias tienen alguna "
              "distancia sin PDF utilizable -- su total solo cuenta las distancias con datos.")

    conteos = (
        _por_grupo.groupby("nombre_evento")[["n_masculino", "n_femenino"]].sum()
        .rename(columns={"n_masculino": "finisher_h", "n_femenino": "finisher_d"})
        .reset_index().rename(columns={"nombre_evento": "nombre_carrera"})
    )
    conteos.loc[conteos["nombre_carrera"].isin(_sin_datos), ["finisher_h", "finisher_d"]] = pd.NA

    curses_limpio = curses_limpio.merge(conteos, on="nombre_carrera", how="left")
    print(f"{RUTA_FINISHERS.name}: {df_finishers_pdf['url_pdf'].nunique()} PDF procesados, "
          f"{conteos.shape[0]} eventos con recuento (canónico por grupo de distancia, sin doble conteo).")
else:
    curses_limpio["finisher_d"] = pd.NA
    curses_limpio["finisher_h"] = pd.NA
    print(f"AVISO: no existe {RUTA_FINISHERS} todavía -- corre la Fase 2 de scrape_youevent.ipynb",
          "en tu máquina (contar_finishers_por_sexo) y vuelve a ejecutar este notebook.")

curses_limpio[["finisher_d", "finisher_h"]].isna().sum()

AVISO: 3 eventos solo tenían PDF de locales/clubes (ningún PDF con el campo completo) -- se quedan sin finisher_d/finisher_h.
AVISO: 19 eventos con varias distancias tienen alguna distancia sin PDF utilizable -- su total solo cuenta las distancias con datos.
DF_YOUEVENT_FINISHERS_SUCIO.csv: 8938 PDF procesados, 1840 eventos con recuento (canónico por grupo de distancia, sin doble conteo).


finisher_d    23
finisher_h    23
dtype: int64

### `distancia`: solo cuando está en el propio nombre del evento

`DF_YOUEVENT_SUCIO.csv` no trae ninguna distancia estructurada (a diferencia de RaceResult) — la única pista es si el nombre del evento la menciona (`"10K Yanguas"`, `"10 kms Villa de Alovera"`...), que pasa en 89 de 1.858 eventos (4,8 %). Para el resto, `distancia = 0`, el mismo valor centinela de "sin distancia" que en ccnorte/mychip/RaceResult.

In [4]:
import re

_RE_DISTANCIA = re.compile(r"(\d+(?:[.,]\d+)?)\s*k(?:m|ms)?\b", re.IGNORECASE)


def _distancia_km(nombre):
    m = _RE_DISTANCIA.search(nombre)
    if not m:
        return 0.0
    return float(m.group(1).replace(",", "."))


curses_limpio["distancia"] = curses_limpio["nombre_carrera"].apply(_distancia_km)
print("Distancia informada (> 0):", (curses_limpio["distancia"] > 0).sum(), "de", len(curses_limpio))
curses_limpio.loc[curses_limpio["distancia"] > 0, ["nombre_carrera", "distancia"]].sample(10, random_state=0)

Distancia informada (> 0): 89 de 1860


,nombre_carrera,distancia
156,GTP110K - GRAN TRAIL PEÑALARA,110.0
273,2ª Carrera Popular de Navidad - CARRERA DE 6 KMS,6.0
720,XVIII Media Maratón de Guadalajara y 11 Km Pop...,11.0
569,XII Media Maratón Ciudad de Cantalejo & 10 Km ...,10.0
972,10 kms Villa de Alovera 2019,10.0
447,"CT 10K ""Cabrera Trail Promo""",10.0
592,5ª Carrera Popular de Navidad de Moralzarzal -...,10.0
589,Cabrera Trail 21K - Media Maratón por Montaña,21.0
1215,XXIII MEDIA MARATON GUADALAJARA Y 11 KM WITZEN...,11.0
1842,IV Carrera Pedestre Popular - 10K Yanguas,10.0


### `tipo_modalidad` y `publico`, por palabras clave en el nombre del evento

Igual que en el resto de fuentes sin una columna de disciplina ya dada: clasificamos por palabras clave, esta vez solo con `nombre_carrera` (no hay ningún otro texto de categoría a nivel de evento en este catálogo). El orden de las reglas importa: duatlón/triatlón se comprueban ANTES que trail, porque nombres como "DUATLON CROS ..." contienen "cross" y si no, caerían mal clasificados.

In [5]:
def _clasificar_tipo_modalidad(nombre):
    texto = nombre.lower()
    if re.search(r"esqu", texto):
        return "Otros"
    if re.search(r"duatl|triatl|triathlon|acuatl|aquathlon", texto):
        return "Multidisciplina"
    if re.search(r"\bbtt\b|\bmtb\b|\bbike\b|ciclis|ciclo", texto):
        return "Ciclismo y btt"
    if re.search(r"trail|cross|vertical|\bkv\b", texto):
        return "trail running"
    if re.search(r"marcha", texto):
        return "marcha"
    return "road running"


curses_limpio["tipo_modalidad"] = curses_limpio["nombre_carrera"].apply(_clasificar_tipo_modalidad)
print(curses_limpio["tipo_modalidad"].value_counts())

tipo_modalidad
road running       826
trail running      526
Multidisciplina    304
marcha             138
Ciclismo y btt      58
Otros                8
Name: count, dtype: int64


In [6]:
def _clasificar_publico(nombre):
    texto = nombre.lower()
    if re.search(r"equipos?\b", texto):
        return "Equipos"
    if re.search(r"veteran|m[aá]ster|\bsenior\b", texto):
        return "Mayores/Veteranos"
    if re.search(r"\belite\b|[ée]lite|profesional", texto):
        return "Elite"
    if re.search(
        r"infantil|alev[ií]n|benjam|cadete|juvenil|j[uú]nior|menores|promesa|escolar",
        texto,
    ):
        return "Infantil"
    return "Absoluta/General"


curses_limpio["publico"] = curses_limpio["nombre_carrera"].apply(_clasificar_publico)
print(curses_limpio["publico"].value_counts())

publico
Absoluta/General     1695
Infantil              153
Equipos                10
Mayores/Veteranos       1
Elite                   1
Name: count, dtype: int64


### Ubicación — geocodificando el nombre del evento (igual que championsxip)

`DF_YOUEVENT_SUCIO.csv` tampoco trae ubicación (ni coordenadas ni texto de lugar aparte del nombre). Reutilizamos la misma heurística de `limpieza_championsxip.ipynb` para aislar un candidato a lugar dentro de `nombre_carrera` (quitar numerales iniciales, año, paréntesis, palabras genéricas de carrera) y geocodificarlo con Nominatim/OpenStreetMap, con checkpoint (`youevent_ubicaciones.csv`) y el mismo aviso: es *best-effort*, no siempre acierta con patrocinadores o instalaciones sin geocodificar en OSM.

A diferencia de championsxip, aquí cacheamos por `candidato_lugar` (el nombre de sitio ya limpiado) en vez de por `nombre_carrera` completo: muchas carreras de este catálogo se repiten cada año con el mismo sitio (948 candidatos únicos para 1.860 eventos), así que agrupar por candidato evita pedirle a Nominatim lo mismo varias veces. Nominatim limita a 1 petición/segundo por política de uso -- a diferencia de la descarga de PDF de `scrape_youevent.ipynb`, esto no se puede paralelizar, así que tarda su rato (~950 peticiones secuenciales, más los reintentos con menos palabras cuando la consulta completa no encuentra nada).</cell id="819ab7ff">

In [ ]:
_RE_NUMERAL_INICIAL = re.compile(
    r"^\s*(?:[ivxlcdm]+|\d+)\s*[ºª]?\.?(?:er|do|ro|a)?\s*[-–]?\s+",
    re.IGNORECASE,
)
_RE_ANIO = re.compile(r"\b(19|20)\d{2}\b")
_RE_DISTANCIA_FINAL = re.compile(r"\s+\d+(?:[.,]\d+)?\s*k(?:m)?\.?$", re.IGNORECASE)

_PALABRAS_GENERICAS = {
    "carrera", "correr", "cross", "cros", "trail", "maraton", "maratón", "media", "medio",
    "milla", "por", "prueba", "equipos", "popular", "urbana", "urbano", "semi", "escolar", "memorial", "trofeo",
    "circuito", "campeonato", "duatlon", "duatlón", "triatlon", "triatlón", "clásica", "clasica", "federada", "federado",
    "nocturna", "nocturno", "solidaria", "solidario", "btt", "ruta", "gran", "premio",
    "edicion", "edición", "subida", "vuelta", "copa", "liga", "internacional",
    "provincial", "final", "comarcal", "nacional", "fase", "previa", "pedestre",
    "ayuntamiento", "villa", "ciudad", "fiesta", "fiestas", "aniversario", "san",
    "silvestre", "vertical", "km", "k", "kv", "kilometro", "kilómetro",
}
_PREPOSICIONES = {"de", "del", "a", "en", "al"}


def _candidato_lugar(nombre_carrera: str) -> str:
    if not isinstance(nombre_carrera, str) or not nombre_carrera.strip():
        return ""

    texto = nombre_carrera.strip()
    texto = _RE_NUMERAL_INICIAL.sub("", texto)
    texto = _RE_ANIO.split(texto)[0]
    texto = texto.split("(")[0]
    texto = _RE_DISTANCIA_FINAL.sub("", texto)
    texto = texto.strip(" -–,.")

    tokens = [token.strip(" ,.-–") for token in texto.split()]
    tokens = [token for token in tokens if token]

    i = 0
    while i < len(tokens) and tokens[i].lower() in _PALABRAS_GENERICAS:
        i += 1
    while i < len(tokens) and tokens[-1].lower() in _PALABRAS_GENERICAS:
        tokens.pop()

    if i < len(tokens) and tokens[i].lower() in _PREPOSICIONES:
        candidato = " ".join(tokens[i + 1:])
    else:
        candidato = " ".join(tokens[i:])

    candidato = candidato.strip(" ,.-–")
    return candidato or texto


# Se guarda como columna (no solo para el print de abajo) porque la
# geocodificación de la siguiente celda hace caché por candidato_lugar, no
# por nombre_carrera: muchas carreras se repiten cada año con el mismo sitio
# (948 candidatos únicos para 1.860 eventos), así que agrupar por candidato
# nos ahorra casi la mitad de las consultas a Nominatim.
curses_limpio["candidato_lugar"] = curses_limpio["nombre_carrera"].apply(_candidato_lugar)

print("Ejemplos nombre_carrera -> candidato_lugar:")
print(
    curses_limpio[["nombre_carrera", "candidato_lugar"]]
    .drop_duplicates()
    .sample(15, random_state=0)
)
print()
print("Candidatos de lugar únicos:", curses_limpio["candidato_lugar"].nunique(),
      "de", curses_limpio["nombre_carrera"].nunique(), "eventos.")

Ejemplos nombre_carrera -> candidato_lugar:
                                         nombre_carrera  \
1224             VI San Silvestre Solidaria ALPEDRETEÑA   
475                 San Silvestre 2015 El Boalo Runners   
1839     Marcha Nordica Montmelo 2026  - Copa de España   
1199                            I Trail Cueva del Beato   
53        DUATLON SALAMANCA ( Cpto. de Castilla y León)   
1408  II Carrera Popular Fiestas Patronales de Talam...   
1654   Gran Triatlón Madrid 2025 - Contrarreloj Equipos   
18       TRIATLON CANAL DE CASTILLA - MEDINA DE RIOSECO   
731             8º Carrera Pedestre Vega de Cantimpalos   
1374                       2ª San Silvestre de Boadilla   
233                        V Cicloturista Pedro Herrero   
1390                       Cross Escolar ADS Final 2024   
621              XIII Carrera Popular Ciudad del Doncel   
427   V Carrera y III Marcha Pedestre Villa de Turégano   
187       6 Horas de Esquí de Fondo Centenario Peñalara   

           

In [8]:
import time as _time


def _geocode_con_reintentos(geolocator, candidato, pausa_segundos):
    from geopy.exc import GeopyError

    tokens = candidato.split()
    max_start = max(0, len(tokens) - 2) if len(tokens) > 1 else 0

    for start in range(max_start + 1):
        query = " ".join(tokens[start:])
        if not query:
            break
        try:
            loc = geolocator.geocode(
                f"{query}, España", exactly_one=True, country_codes="es",
                addressdetails=True, timeout=10,
            )
        except GeopyError:
            loc = None
        if loc:
            return loc
        if start < max_start:
            _time.sleep(pausa_segundos)
    return None


def geocodificar_ubicaciones_youevent(candidatos_lugar, out_dir, pausa_segundos: float = 1.1):
    """Geocodifica cada CANDIDATO A LUGAR único (no cada nombre_carrera): muchas
    carreras se repiten cada año con el mismo sitio (948 candidatos únicos para
    1.860 eventos), así que cachear por candidato en vez de por nombre completo
    evita pedir lo mismo dos veces. Nominatim limita a 1 petición/segundo por
    política de uso -- no se puede paralelizar como hicimos con los PDF de
    youevent, así que esto tarda su rato (~950 peticiones secuenciales).

    Dos detalles que costaron descubrir a base de un intento fallido real:
    - En instalaciones de Python de python.org en Mac, el almacén de
      certificados SSL del propio Python puede no estar enlazado (no es un
      problema de red ni de Nominatim): todas las peticiones fallan con
      'certificate verify failed' que geopy traga como GeopyError y convierte
      en "no encontrado" -- silenciosamente, sin ningún error visible, así que
      puede parecer que la geocodificación "funciona" y en realidad falla al
      100 %. Se arregla fijando el contexto SSL con el certificado de
      `certifi` antes de crear el geolocator.
    - Se reescribe el CSV entero desde el diccionario `cache` (no se va
      añadiendo fila a fila con 'a'): si esta celda se llega a ejecutar dos
      veces a la vez (p. ej. una vez desde este notebook y otra desde un
      script suelto), añadir filas sueltas duplica candidatos en el fichero;
      reescribir desde `cache` (clave = candidato_lugar) es idempotente pase
      lo que pase."""
    import ssl

    import certifi
    import geopy.geocoders
    from geopy.geocoders import Nominatim

    geopy.geocoders.options.default_ssl_context = ssl.create_default_context(cafile=certifi.where())

    out_path = Path(out_dir)
    csv_ubic = out_path / "youevent_ubicaciones.csv"
    campos = ["candidato_lugar", "municipio", "comarca", "provincia", "lat", "lon"]

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["candidato_lugar"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} candidatos ya geocodificados")

    geolocator = Nominatim(user_agent="youevent_limpieza_claudia")

    candidatos_unicos = list(dict.fromkeys(c for c in candidatos_lugar if isinstance(c, str) and c))
    pendientes = [c for c in candidatos_unicos if c not in cache]
    print(f"Candidatos a geocodificar: {len(pendientes)} (de {len(candidatos_unicos)} únicos)")

    for i, candidato in enumerate(pendientes, 1):
        fila = {c: None for c in campos}
        fila["candidato_lugar"] = candidato
        try:
            loc = _geocode_con_reintentos(geolocator, candidato, pausa_segundos)
            if loc:
                addr = loc.raw.get("address", {})
                fila["municipio"] = (
                    addr.get("city") or addr.get("town") or addr.get("village")
                    or addr.get("municipality")
                )
                fila["comarca"] = addr.get("county")
                fila["provincia"] = addr.get("province") or addr.get("state")
                fila["lat"] = loc.latitude
                fila["lon"] = loc.longitude
        except Exception as exc:
            print(f"  [{i}/{len(pendientes)}] error geocodificando \"{candidato}\": {exc}")

        cache[candidato] = fila
        if i % 50 == 0:
            pd.DataFrame(cache.values())[campos].to_csv(csv_ubic, index=False)
            print(f"  [{i}/{len(pendientes)}] geocodificados (checkpoint guardado)")
        _time.sleep(pausa_segundos)

    df_resultado = pd.DataFrame(cache.values())[campos]
    df_resultado.to_csv(csv_ubic, index=False)
    return df_resultado


OUT_DIR = Path("../../data/raw/youevent")
df_ubicaciones = geocodificar_ubicaciones_youevent(curses_limpio["candidato_lugar"], out_dir=OUT_DIR)
curses_limpio = curses_limpio.merge(
    df_ubicaciones[["candidato_lugar", "municipio", "comarca", "provincia"]],
    on="candidato_lugar", how="left",
)
print()
print("Eventos con municipio identificado:", curses_limpio["municipio"].notna().sum(),
      "de", len(curses_limpio))

Checkpoint: 948 candidatos ya geocodificados
Candidatos a geocodificar: 0 (de 948 únicos)

Eventos con municipio identificado: 971 de 1860


### Esquema común entre las fuentes

Igual que en el resto de fuentes del proyecto, las 12 columnas compartidas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante (`"youevent"`). Lo propio de youevent (`tiene_sexo_identificado`, `n_enlaces_sexo`) va al final, como recordatorio de que `finisher_d`/`finisher_h`/`municipio`/`comarca`/`provincia` están todavía sin rellenar a la espera de correr en local la descarga de PDF y la geocodificación.

In [9]:
curses_limpio["fuente"] = "youevent"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

_COLUMNAS_COMUNES = [
    "fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
    "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
]
_COLUMNAS_PROPIAS = [c for c in curses_limpio.columns if c not in _COLUMNAS_COMUNES]
curses_limpio = curses_limpio[_COLUMNAS_COMUNES + _COLUMNAS_PROPIAS]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'n_enlaces_sexo',
 'tiene_sexo_identificado',
 'candidato_lugar']

In [10]:
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
print("Recordatorio: finisher_d/finisher_h se rellenan solos si ya has corrido la Fase 2 de",
      "scrape_youevent.ipynb (contar_finishers_por_sexo) -- si no, quedan en NaN. municipio/",
      "comarca/provincia se rellenan con geocodificar_ubicaciones_youevent() (más arriba) --",
      "puede quedar algún NaN suelto cuando Nominatim no encuentra el candidato a lugar.")
curses_limpio.sample(10, random_state=0)

Filas x columnas: (1860, 15)

fuente                             object
nombre_carrera                     object
fecha                      datetime64[ns]
dia_semana                         object
distancia                         float64
tipo_modalidad                     object
publico                            object
finisher_d                        float64
finisher_h                        float64
municipio                          object
comarca                            object
provincia                          object
n_enlaces_sexo                      int64
tiene_sexo_identificado              bool
candidato_lugar                    object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Elite  Equipos  Infantil  Mayores/Veteranos
tipo_modalidad                                                                
Ciclismo y btt                 56      0        0         2                  0
Multidisciplina               234      0        4       

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,n_enlaces_sexo,tiene_sexo_identificado,candidato_lugar
1224,youevent,VI San Silvestre Solidaria ALPEDRETEÑA,2022-12-31,Sábado,0.0,road running,Absoluta/General,94.0,152.0,NaN,NaN,NaN,0,False,ALPEDRETEÑA
475,youevent,San Silvestre 2015 El Boalo Runners,2015-12-26,Sábado,0.0,road running,Absoluta/General,95.0,143.0,Santiago de Compostela,Santiago,Galicia,0,False,San Silvestre
1839,youevent,Marcha Nordica Montmelo 2026 - Copa de España,2026-07-18,Sábado,0.0,marcha,Absoluta/General,58.0,77.0,NaN,NaN,NaN,4,True,Marcha Nordica Montmelo
1199,youevent,I Trail Cueva del Beato,2022-10-23,Domingo,0.0,trail running,Absoluta/General,10.0,41.0,Cifuentes,NaN,Castilla-La Mancha,0,False,Cueva del Beato
53,youevent,DUATLON SALAMANCA ( Cpto. de Castilla y León),2010-04-25,Domingo,0.0,Multidisciplina,Absoluta/General,10.0,102.0,Salamanca,NaN,Castilla y León,2,True,SALAMANCA
1408,youevent,II Carrera Popular Fiestas Patronales de Talam...,2024-03-23,Sábado,0.0,road running,Absoluta/General,0.0,0.0,El Casar,NaN,Castilla-La Mancha,0,False,Patronales de Talamanca
1654,youevent,Gran Triatlón Madrid 2025 - Contrarreloj Equipos,2025-06-22,Domingo,0.0,Multidisciplina,Equipos,0.0,0.0,Madrid,NaN,Comunidad de Madrid,6,True,Madrid
18,youevent,TRIATLON CANAL DE CASTILLA - MEDINA DE RIOSECO,2008-08-16,Sábado,0.0,Multidisciplina,Absoluta/General,0.0,87.0,Medina de Rioseco,NaN,Castilla y León,2,True,CANAL DE CASTILLA - MEDINA DE RIOSECO
731,youevent,8º Carrera Pedestre Vega de Cantimpalos,2018-02-11,Domingo,0.0,road running,Absoluta/General,3.0,5.0,NaN,NaN,NaN,0,False,Vega de Cantimpalos
1374,youevent,2ª San Silvestre de Boadilla,2023-12-16,Sábado,0.0,road running,Absoluta/General,0.0,0.0,La Fuente de San Esteban,NaN,Castilla y León,0,False,Boadilla


In [11]:
SALIDA = Path("../../data/processed/youevent/DF_YOUEVENT_LIMPIO.csv")
SALIDA.parent.mkdir(parents=True, exist_ok=True)
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en ../../data/processed/youevent/DF_YOUEVENT_LIMPIO.csv
